# Mini Transformer Integration Workshop Lab

Assemble embeddings, positions, causal blocks, final normalization, and a vocabulary head. Prove prefix causality and run a backward pass.


In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(16)

class Block(nn.Module):
    def __init__(self,width,heads):
        super().__init__()
        if width % heads: raise ValueError("n_embd must be divisible by n_head")
        self.h=heads; self.d=width//heads
        self.qkv=nn.Linear(width,3*width,bias=False)
        self.proj=nn.Linear(width,width,bias=False)
        self.ln1=nn.LayerNorm(width); self.ln2=nn.LayerNorm(width)
        self.ff=nn.Sequential(nn.Linear(width,4*width),nn.GELU(),nn.Linear(4*width,width))
    def attention(self,x):
        B,T,C=x.shape
        q,k,v=self.qkv(x).chunk(3,-1)
        def h(z): return z.view(B,T,self.h,self.d).transpose(1,2)
        q,k,v=h(q),h(k),h(v)
        logits=q@k.transpose(-2,-1)/math.sqrt(self.d)
        mask=torch.triu(torch.ones(T,T,dtype=torch.bool,device=x.device),diagonal=1)
        w=torch.softmax(logits.masked_fill(mask,float("-inf")),dim=-1)
        y=(w@v).transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)
    def forward(self,x):
        x=x+self.attention(self.ln1(x))
        return x+self.ff(self.ln2(x))

class MiniTransformer(nn.Module):
    def __init__(self,vocab_size=23,context=8,width=16,heads=4,depth=2):
        super().__init__()
        self.context=context
        self.token=nn.Embedding(vocab_size,width)
        self.pos=nn.Embedding(context,width)
        self.blocks=nn.ModuleList([Block(width,heads) for _ in range(depth)])
        self.norm=nn.LayerNorm(width)
        self.head=nn.Linear(width,vocab_size,bias=False)
    def forward(self,ids):
        B,T=ids.shape
        if T>self.context: raise ValueError("sequence exceeds context")
        positions=torch.arange(T,device=ids.device)
        x=self.token(ids)+self.pos(positions)[None,:,:]
        for block in self.blocks: x=block(x)
        return self.head(self.norm(x))

model=MiniTransformer()
ids=torch.tensor([[1,2,3,4,5]])
alt=torch.tensor([[1,2,3,9,10]])
logits=model(ids)
alt_logits=model(alt)
assert logits.shape==(1,5,23)
assert torch.allclose(logits[:,:3],alt_logits[:,:3],atol=1e-6), "future tokens changed earlier logits"
loss=logits.square().mean()
loss.backward()
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
print("PASS: l05-16 mini transformer checks")
